<a href="https://colab.research.google.com/github/gapr97/aoe2-civ-explorer/blob/main/scripts/01_build_tables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd

USER = "gapr97"
REPO = "aoe2-civ-explorer"
BASE = f"https://raw.githubusercontent.com/{USER}/{REPO}/main/data/source"

def get_json(path):
    r = requests.get(f"{BASE}/{path}")
    r.raise_for_status()
    return r.json()

def describe(obj, name="root", depth=0, max_depth=3):
    """Print the structure of a JSON object without dumping all of it."""
    pad = "  " * depth
    if isinstance(obj, dict):
        print(f"{pad}{name}: dict, {len(obj)} keys -> {list(obj.keys())[:8]}")
        if depth < max_depth:
            for k in list(obj.keys())[:5]:
                describe(obj[k], k, depth + 1, max_depth)
    elif isinstance(obj, list):
        print(f"{pad}{name}: list, {len(obj)} items")
        if obj and depth < max_depth:
            describe(obj[0], "[0]", depth + 1, max_depth)
    else:
        print(f"{pad}{name}: {type(obj).__name__} = {str(obj)[:60]}")

# Load the main files
data = get_json("data.json")
strings = get_json("locales/en/strings.json")

print("===== data.json =====")
describe(data, "data.json")
print("\n===== strings.json =====")
describe(strings, "strings.json", max_depth=1)

# List the civ tree files and peek at one
api = f"https://api.github.com/repos/{USER}/{REPO}/contents/data/source/trees"
tree_files = [f["name"] for f in requests.get(api).json()]
print(f"\n===== trees: {len(tree_files)} files =====")
print(tree_files[:5])
describe(get_json(f"trees/{tree_files[0]}"), tree_files[0])

===== data.json =====
data.json: dict, 4 keys -> ['age_names', 'civs', 'data', 'tech_tree_strings']
  age_names: dict, 2 keys -> ['antiquity', 'base']
    antiquity: list, 4 items
      [0]: str = 407089
    base: list, 4 items
      [0]: str = 4201
  civs: dict, 56 keys -> ['Armenians', 'Aztecs', 'Bengalis', 'Berbers', 'Bohemians', 'Britons', 'Bulgarians', 'Burgundians']
    Armenians: dict, 8 keys -> ['Building', 'Tech', 'Unit', 'era', 'help_string_id', 'internal_name', 'meta', 'name_string_id']
      Building: list, 27 items
      Tech: list, 74 items
      Unit: list, 49 items
      era: str = base
      help_string_id: int = 120193
    Aztecs: dict, 8 keys -> ['Building', 'Tech', 'Unit', 'era', 'help_string_id', 'internal_name', 'meta', 'name_string_id']
      Building: list, 26 items
      Tech: list, 66 items
      Unit: list, 44 items
      era: str = base
      help_string_id: int = 120164
    Bengalis: dict, 8 keys -> ['Building', 'Tech', 'Unit', 'era', 'help_string_id', 'int

In [2]:
# AoE2 Civ Explorer - build clean tables from the aoe2techtree data
# Paste this whole file into one Colab cell and run it (Shift+Enter).

import os, re, requests
import pandas as pd

USER = "gapr97"
REPO = "aoe2-civ-explorer"
BASE = f"https://raw.githubusercontent.com/{USER}/{REPO}/main/data/source"

def get_json(path):
    r = requests.get(f"{BASE}/{path}")
    r.raise_for_status()
    return r.json()

data = get_json("data.json")
strings = get_json("locales/en/strings.json")

def txt(string_id):
    """Look up an English text by its string id."""
    return strings.get(str(string_id), "")

def clean(s):
    return re.sub(r"<[^>]+>", "", s).strip(" •\n")

# ---------- 1. Civs + bonuses (parsed from the civ help text) ----------
civ_rows, bonus_rows = [], []
for civ, c in data["civs"].items():
    lines = [l.strip() for l in txt(c["help_string_id"]).split("<br>")]
    lines = [l for l in lines if l]
    specialty = clean(lines[0])
    section = "Civ Bonus"
    unique_units = team_bonus = ""
    for l in lines[1:]:
        if "Unique Unit" in l:
            section = "Unique Unit"; continue
        if "Unique Tech" in l:
            section = "Unique Tech"; continue
        if "Team Bonus" in l:
            section = "Team Bonus"; continue
        text = clean(l)
        if not text:
            continue
        if section == "Unique Unit":
            unique_units = text
        else:
            if section == "Team Bonus":
                team_bonus = text
            bonus_rows.append({"civ": civ, "type": section, "text": text})
    civ_rows.append({"civ": civ, "specialty": specialty,
                     "unique_units": unique_units, "team_bonus": team_bonus})

dim_civ = pd.DataFrame(civ_rows)
civ_bonus = pd.DataFrame(bonus_rows)
civ_bonus.insert(0, "bonus_id", range(1, len(civ_bonus) + 1))
# empty columns for you to fill in by hand later (Phase 4)
civ_bonus["category"] = ""        # Eco / Military
civ_bonus["line_affected"] = ""   # Archery, Cavalry, Food, Wood...

# ---------- 2. Units ----------
unit_rows, attack_rows, armour_rows = [], [], []
for uid, u in data["data"]["Unit"].items():
    cost = u.get("Cost", {})
    unit_rows.append({
        "unit_id": int(uid),
        "name": txt(u.get("LanguageNameId")) or u.get("internal_name"),
        "internal_name": u.get("internal_name"),
        "hp": u.get("HP"), "attack": u.get("Attack"),
        "melee_armor": u.get("MeleeArmor"), "pierce_armor": u.get("PierceArmor"),
        "range": u.get("Range"), "reload_time": u.get("ReloadTime"),
        "speed": u.get("Speed"), "line_of_sight": u.get("LineOfSight"),
        "train_time": u.get("TrainTime"), "accuracy_pct": u.get("AccuracyPercent"),
        "food": cost.get("Food", 0), "wood": cost.get("Wood", 0),
        "gold": cost.get("Gold", 0), "stone": cost.get("Stone", 0),
    })
    for a in u.get("Attacks", []):
        attack_rows.append({"unit_id": int(uid), "class_id": a["Class"], "amount": a["Amount"]})
    for a in u.get("Armours", []):
        armour_rows.append({"unit_id": int(uid), "class_id": a["Class"], "amount": a["Amount"]})

dim_unit = pd.DataFrame(unit_rows)
dim_unit["total_cost"] = dim_unit[["food", "wood", "gold", "stone"]].sum(axis=1)
unit_attacks = pd.DataFrame(attack_rows)
unit_armours = pd.DataFrame(armour_rows)

# ---------- 3. Techs ----------
tech_rows = []
for tid, t in data["data"]["Tech"].items():
    cost = t.get("Cost", {})
    tech_rows.append({
        "tech_id": int(tid),
        "name": txt(t.get("LanguageNameId")) or t.get("internal_name"),
        "research_time": t.get("ResearchTime"),
        "food": cost.get("Food", 0), "wood": cost.get("Wood", 0),
        "gold": cost.get("Gold", 0), "stone": cost.get("Stone", 0),
    })
dim_tech = pd.DataFrame(tech_rows)

# ---------- 4. Tech tree per civ (what each civ has and lacks) ----------
AGES = {1: "Dark", 2: "Feudal", 3: "Castle", 4: "Imperial"}
tree_rows, names = [], {}
for civ in data["civs"]:
    tree = get_json(f"trees/{civ.upper()}.json")
    buildings = {b["node_id"]: b["name"] for b in tree["buildings"]}
    for n in tree["units_techs"]:
        names[(n["use_type"], n["node_id"])] = n["name"]
        tree_rows.append({
            "civ": civ,
            "node_id": n["node_id"],
            "name": n["name"],
            "kind": n["use_type"],            # Unit / Tech / Building
            "node_type": n["node_type"],      # Unit, UnitUpgrade, UniqueUnit, Research...
            "building": buildings.get(n["building_id"], ""),
            "age": AGES.get(n["age_id"], ""),
            "available": 0 if n["node_status"] == "NotAvailable" else 1,
        })
civ_tech_tree = pd.DataFrame(tree_rows)

# use the proper English names from the trees
dim_unit["name"] = [names.get(("Unit", i), n) for i, n in zip(dim_unit.unit_id, dim_unit.name)]
dim_tech["name"] = [names.get(("Tech", i), n) for i, n in zip(dim_tech.tech_id, dim_tech.name)]

# ---------- 5. Save ----------
tables = {
    "dim_civ": dim_civ, "civ_bonus": civ_bonus, "dim_unit": dim_unit,
    "unit_attacks": unit_attacks, "unit_armours": unit_armours, "dim_tech": dim_tech,
    "civ_tech_tree": civ_tech_tree,
}
os.makedirs("processed", exist_ok=True)
for name, df in tables.items():
    df.to_csv(f"processed/{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name:16} {len(df):>6} rows")

print("\nSample - Armenian bonuses:")
print(civ_bonus[civ_bonus.civ == "Armenians"][["type", "text"]].to_string(index=False))
print("\nSample - what Armenians are missing:")
missing = civ_tech_tree[(civ_tech_tree.civ == "Armenians") & (civ_tech_tree.available == 0)]
print(missing[["name", "building", "age"]].to_string(index=False))

# ---------- 6. Download the CSVs as one zip (Colab only) ----------
import shutil
shutil.make_archive("processed", "zip", "processed")
try:
    from google.colab import files
    files.download("processed.zip")
except ImportError:
    print("\nNot running in Colab - CSVs are in the 'processed' folder.")

dim_civ              56 rows
civ_bonus           399 rows
dim_unit            255 rows
unit_attacks       1639 rows
unit_armours       1343 rows
dim_tech            201 rows
civ_tech_tree      8158 rows

Sample - Armenian bonuses:
       type                                                                                  text
  Civ Bonus                                                                  Mule Carts cost -25%
  Civ Bonus                                                     Mule Cart technology effects +40%
  Civ Bonus    Spearman- and Militia-line upgrades (except Man-at-Arms) available one age earlier
  Civ Bonus                                          First Fortified Church receives a free Relic
  Civ Bonus                                 Galley-line and Dromons fire an additional projectile
Unique Tech Cilician Fleet (Demolition Ships +20% blast radius; Galley-line and Dromons +1 range)
Unique Tech  Fereters (Infantry (except Spearman-line) +30 HP; Warrior Priests heal

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>